# OOD Testing: Where Could the Pipeline Beat MILP?

The previous matched-budget evaluation (`time_constrained_eval.ipynb`) showed that on the
in-distribution low/medium/high families:

- **MILP time-to-first-feasible is 9-45x faster than the pipeline's wall-clock.**
- **`pipeline_only` count = 0** in every family.
- **`pipe_first_pct = 0%`** in every family (the pipeline never finishes before MILP loads an incumbent).

This rules out a pipeline win on the existing distribution. This notebook tests three concrete OOD
regimes that *could* in principle invert that ranking. Each experiment is independent and self-contained
behind a `RUN_*` flag.

## Hypotheses

**H-A (Structural-scale OOD).** On scenarios with much larger topology than training
(many regions x many zones x many thermal/nuclear units, T = 24 unchanged so the encoder still works),
MILP's time-to-first-feasible explodes faster than the pipeline's wall-clock. The pipeline should
produce non-zero `pipeline_only` cells if and only if MILP TTF starts to exceed pipeline wall-clock
on these instances.

**H-B (Adversarial UC structure).** On scenarios with intentionally weak LP relaxations
(tight ramps, low storage/DR, high VOLL, expensive startups, congested transmission), MILP cannot find
a feasible incumbent quickly even at moderate problem size. This is the regime where, regardless of
scale, the pipeline's projection-LP could be faster than MILP branch-and-cut.

**H-C (Lean-pipeline at extreme budgets).** On the *existing* low/medium/high scenarios, a stripped
down pipeline (K=1, 20 Langevin steps, hard_fix only) might fit in 1-2 s wall-clock. At those budgets,
MILP cannot even finish presolve on most scenarios. This is a *pipeline-only* niche purchased by
trading solution quality and feasibility-recovery depth for speed.

## Reading caveats

- All three experiments **keep the encoder/EBM unchanged from training** (T = 24, n_features = 7,
  same asset types). They isolate the MIP-hardness axis from any encoder-generalization confound.
- Total compute is heavy (scenario generation + reference MILP at uncapped time + pipeline + matched
  MILP + lean pipeline). Defaults below cap each experiment at ~10-20 scenarios. Bump them once you
  have a working baseline.
- The pipeline runs here use the **full** repair cascade (`enable_exact_fallback=False` only on
  Experiment C's lean config) since the question is whether the *deployable* pipeline can win, not
  whether the EBM alone can.

## 0. Imports and Repo Discovery

In [ ]:
import json
import math
import os
import pickle
import shutil
import subprocess
import sys
import time
from dataclasses import asdict
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Dict, List, Optional, Tuple

_cwd = Path.cwd().resolve()
_repo_candidates = [_cwd, *_cwd.parents]
for _cand in list(_repo_candidates):
    _nested = _cand / 'benchmark_milp_gnn'
    if (_nested / 'src').exists():
        _repo_candidates.append(_nested)
for _cand in _repo_candidates:
    if (_cand / 'src').exists() and (_cand / 'config').exists():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        REPO_PATH = _cand
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm

# Force-reload of repo modules so source edits propagate.
for module_name in [m for m in list(sys.modules) if m.startswith('src.')]:
    del sys.modules[module_name]

from src.milp.solve import solve_scenario
from src.eval.metrics import build_comparison_dataframe, load_milp_reports, compute_eval_metrics
from src.eval.pipeline_runner import (
    PipelineConfig,
    PipelineResult,
    PipelineRunner,
    backfill_projection_diagnostics,
)

sns.set_theme(style='whitegrid')
sns.set_context('talk', font_scale=0.85)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Repo: {REPO_PATH}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)}')

## 1. Configuration

All paths and run-control flags. Set `RUN_*` flags to `True` for the experiments you want
executed end-to-end; set them to `False` once you have a cache to load.

In [ ]:
OOD_OUTPUT_DIR = REPO_PATH / 'outputs' / 'ood_testing'
OOD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Reference checkpoints (same as end_evaluation_ebm_v3_good) ---
EBM_CHECKPOINT_REL = 'outputs/ebm_models/ebm_v3_good/ebm_v3_final.pt'
ENCODER_CHECKPOINT_REL = 'outputs/encoders/hierarchical_temporal_v3/best_encoder.pt'
assert (REPO_PATH / EBM_CHECKPOINT_REL).exists(), REPO_PATH / EBM_CHECKPOINT_REL
assert (REPO_PATH / ENCODER_CHECKPOINT_REL).exists(), REPO_PATH / ENCODER_CHECKPOINT_REL

# --- Per-experiment OOD scenario layout ---
OOD_FAMILIES = {
    'hard': {
        'config_yaml': REPO_PATH / 'config' / 'scenario_space_hard.yaml',
        'scenarios_dir': OOD_OUTPUT_DIR / 'scenarios_hard',
        'reports_dir':   OOD_OUTPUT_DIR / 'scenarios_hard' / 'reports',
        'pipeline_pkl':  OOD_OUTPUT_DIR / 'pipeline_hard.pkl',
        'tc_milp_pkl':   OOD_OUTPUT_DIR / 'tc_milp_hard.pkl',
    },
    'adversarial': {
        'config_yaml': REPO_PATH / 'config' / 'scenario_space_adversarial.yaml',
        'scenarios_dir': OOD_OUTPUT_DIR / 'scenarios_adversarial',
        'reports_dir':   OOD_OUTPUT_DIR / 'scenarios_adversarial' / 'reports',
        'pipeline_pkl':  OOD_OUTPUT_DIR / 'pipeline_adversarial.pkl',
        'tc_milp_pkl':   OOD_OUTPUT_DIR / 'tc_milp_adversarial.pkl',
    },
}

# --- In-distribution scenarios reused by Experiment C ---
IND_FAMILIES = {
    'low':    REPO_PATH / 'outputs' / 'low_criticality_scenarios',
    'medium': REPO_PATH / 'outputs' / 'medium_criticality_scenarios',
    'high':   REPO_PATH / 'outputs' / 'high_criticality_scenarios',
}
EXISTING_PIPELINE_PKL = REPO_PATH / 'outputs' / 'end_evaluation_ebm_v3_good' / 'pipeline_eval_all_families.pkl'

# --- Run-control flags ---
RUN_GENERATION   = False   # Generate OOD scenarios via subprocess to src.generator.run_generator
RUN_REFERENCE_MILP = False  # Solve every OOD scenario with a generous time limit (REFERENCE_TIME_LIMIT)
RUN_PIPELINE_OOD = False   # Run the full pipeline on hard + adversarial scenarios
RUN_MATCHED_MILP = False   # Run time-constrained MILP at pipeline wall-clock per OOD scenario
RUN_LEAN_PIPELINE = False  # Experiment C: lean pipeline on in-distribution scenarios
RUN_LEAN_MILP    = False   # Experiment C: tiny-budget MILP on in-distribution scenarios

# --- Budgets ---
REFERENCE_TIME_LIMIT_SECONDS = 1200.0          # cap for the OOD reference MILP runs
FIXED_BUDGETS_SECONDS = [1.0, 5.0, 30.0, 120.0, 300.0]  # for Experiments A/B
LEAN_FIXED_BUDGETS_SECONDS = [0.5, 1.0, 2.0, 5.0]       # for Experiment C
MAX_OOD_SCENARIOS_PER_FAMILY = 12              # keep small at first
MIN_MATCHED_BUDGET_SECONDS = 0.25

SOLVER_NAME = 'appsi_highs'
TEE_SOLVER = False

print('OOD output dir:', OOD_OUTPUT_DIR)
for fam, paths in OOD_FAMILIES.items():
    print(f'  {fam}: yaml={paths["config_yaml"].name}  scenarios={paths["scenarios_dir"].exists()}  reports={paths["reports_dir"].exists()}')
print(f'In-dist pipeline cache exists: {EXISTING_PIPELINE_PKL.exists()}')

## 2. Helpers

Subset of helpers from `time_constrained_eval.ipynb` (kept inline so the notebook is self-contained).
Includes `summarize_milp_run`, `run_constrained_milp_with_guard`, and a re-implementation of
`build_budget_summary` that handles per-scenario matched budgets correctly.

In [ ]:
def _safe_float(value, default=float('nan')):
    try:
        v = float(value)
    except (TypeError, ValueError):
        return default
    return v if np.isfinite(v) else default

def _safe_gap_pct(objective, reference):
    o, r = _safe_float(objective), _safe_float(reference)
    if not np.isfinite(o) or not np.isfinite(r):
        return float('nan')
    return 100.0 * (o - r) / max(abs(r), 1e-9)

def _termination_str(term):
    if term is None:
        return ''
    if hasattr(term, 'name'):
        return str(term.name)
    return str(term)

def _is_timeout_term(term):
    label = _termination_str(term).strip().lower()
    norm = ''.join(ch for ch in label if ch.isalnum())
    return norm in {'maxtimelimit', 'timelimit', 'maxtime', 'timelimitreached', 'maxruntime', 'maxiterations'}


def run_constrained_milp_with_guard(scenario_path: Path, budget_seconds: float):
    t0 = time.perf_counter()
    try:
        payload = solve_scenario(
            scenario_path=scenario_path,
            solver_name=SOLVER_NAME,
            tee=TEE_SOLVER,
            capture_detail=False,
            warm_start_data=None,
            time_limit_seconds=float(budget_seconds),
        )
    except RuntimeError as exc:
        elapsed = time.perf_counter() - t0
        msg = str(exc).lower()
        if 'feasible solution was not found' in msg or 'no solution can be loaded' in msg:
            payload = {
                'scenario_id': scenario_path.stem,
                'mip': SimpleNamespace(
                    objective=float('inf'),
                    termination_condition='no_feasible_solution_loaded',
                    solver_status='error',
                    solve_seconds=elapsed,
                ),
                'lp': SimpleNamespace(
                    objective=float('nan'),
                    termination_condition='skipped',
                    solver_status='skipped',
                    solve_seconds=float('nan'),
                ),
                'cost_components': {},
                'detail': None,
                'feasibility_status': 'no_feasible_solution_loaded',
                'scenario_classification': 'time_constrained_failure',
                'exception_message': str(exc),
                'time_limit_seconds': float(budget_seconds),
            }
            return payload, elapsed
        raise
    return payload, time.perf_counter() - t0


def summarize_milp_run(payload, *, family, scenario_id, criticality_index, reference_objective,
                       reference_solve_time, budget_mode, budget_label, budget_seconds, end_to_end_seconds,
                       pipeline_row=None):
    mip = payload.get('mip')
    lp = payload.get('lp')
    mip_obj_raw = _safe_float(getattr(mip, 'objective', float('nan')))
    feasible = np.isfinite(mip_obj_raw) and not math.isinf(mip_obj_raw)
    mip_obj = mip_obj_raw if feasible else float('nan')
    mip_term = _termination_str(getattr(mip, 'termination_condition', ''))
    gap = _safe_gap_pct(mip_obj, reference_objective)
    row = {
        'scenario_id': scenario_id, 'family': family,
        'criticality_index': criticality_index,
        'budget_mode': budget_mode, 'budget_label': budget_label,
        'budget_seconds': float(budget_seconds),
        'reference_milp_objective': reference_objective,
        'reference_milp_solve_time': reference_solve_time,
        'milp_found_feasible': bool(feasible),
        'milp_optimal': mip_term.lower() == 'optimal',
        'milp_timed_out': _is_timeout_term(mip_term),
        'milp_termination': mip_term,
        'milp_objective': mip_obj,
        'milp_gap_vs_reference_pct': gap,
        'milp_abs_gap_vs_reference_pct': abs(gap) if np.isfinite(gap) else float('nan'),
        'milp_mip_solve_time_s': _safe_float(getattr(mip, 'solve_seconds', float('nan'))),
        'milp_lp_relax_time_s': _safe_float(getattr(lp, 'solve_seconds', float('nan'))),
        'milp_end_to_end_time_s': float(end_to_end_seconds),
    }
    if pipeline_row is not None:
        po = _safe_float(pipeline_row.get('pipeline_objective', float('nan')))
        pt = _safe_float(pipeline_row.get('pipeline_solve_time', float('nan')))
        pgap = _safe_gap_pct(po, reference_objective)
        row.update({
            'pipeline_found_feasible': bool(np.isfinite(po)),
            'pipeline_within_budget': bool(np.isfinite(pt) and pt <= float(budget_seconds) + 1e-9),
            'pipeline_feasible_within_budget': bool(np.isfinite(po) and np.isfinite(pt) and pt <= float(budget_seconds) + 1e-9),
            'pipeline_objective': po,
            'pipeline_solve_time': pt,
            'pipeline_gap_vs_reference_pct': pgap,
            'pipeline_abs_gap_vs_reference_pct': abs(pgap) if np.isfinite(pgap) else float('nan'),
            'pipeline_best_status': str(pipeline_row.get('pipeline_best_status', '')),
            'pipeline_stage_reached': str(pipeline_row.get('pipeline_stage_reached', '')),
        })
    return row


def _aggregate(budget_df, family, budget_mode, budget_seconds):
    both = budget_df['pipeline_feasible_within_budget'] & budget_df['milp_found_feasible']
    pipe_better = both & (budget_df['pipeline_abs_gap_vs_reference_pct'] <= budget_df['milp_abs_gap_vs_reference_pct'])
    milp_better = both & (budget_df['milp_abs_gap_vs_reference_pct'] < budget_df['pipeline_abs_gap_vs_reference_pct'])
    pf = budget_df.loc[budget_df['pipeline_feasible_within_budget'], 'pipeline_abs_gap_vs_reference_pct']
    mf = budget_df.loc[budget_df['milp_found_feasible'], 'milp_abs_gap_vs_reference_pct']
    return {
        'family': family, 'budget_mode': budget_mode, 'budget_seconds': float(budget_seconds),
        'n_scenarios': int(len(budget_df)),
        'pipeline_feasible_rate_pct': float(100.0 * budget_df['pipeline_feasible_within_budget'].mean()),
        'milp_feasible_rate_pct': float(100.0 * budget_df['milp_found_feasible'].mean()),
        'both_feasible_rate_pct': float(100.0 * both.mean()),
        'pipeline_only_rate_pct': float(100.0 * (budget_df['pipeline_feasible_within_budget'] & ~budget_df['milp_found_feasible']).mean()),
        'milp_only_rate_pct': float(100.0 * (~budget_df['pipeline_feasible_within_budget'] & budget_df['milp_found_feasible']).mean()),
        'pipeline_median_abs_gap_pct': float(pf.median()) if len(pf) else float('nan'),
        'milp_median_abs_gap_pct': float(mf.median()) if len(mf) else float('nan'),
        'pipeline_better_gap_rate_pct': float(100.0 * pipe_better.mean()),
        'milp_better_gap_rate_pct': float(100.0 * milp_better.mean()),
        'median_pipeline_time_s': float(budget_df['pipeline_solve_time'].median()),
    }

def build_budget_summary(df, budget_mode):
    scope = df[df['budget_mode'] == budget_mode].copy()
    if not len(scope):
        return pd.DataFrame()
    rows = []
    families = list(scope['family'].unique()) + ['ALL']
    for family in families:
        fam_df = scope if family == 'ALL' else scope[scope['family'] == family]
        if not len(fam_df):
            continue
        if budget_mode == 'fixed':
            for bs, bdf in fam_df.groupby('budget_seconds', sort=True):
                rows.append(_aggregate(bdf, family, budget_mode, float(bs)))
        else:
            r = _aggregate(fam_df, family, budget_mode, float(fam_df['budget_seconds'].median()))
            r.update({
                'budget_seconds_median': float(fam_df['budget_seconds'].median()),
                'budget_seconds_p10': float(fam_df['budget_seconds'].quantile(0.10)),
                'budget_seconds_p90': float(fam_df['budget_seconds'].quantile(0.90)),
            })
            rows.append(r)
    out = pd.DataFrame(rows)
    if budget_mode == 'fixed':
        return out.sort_values(['family', 'budget_seconds']).reset_index(drop=True)
    return out.reset_index(drop=True)

## 3. Generate OOD Scenarios (Experiments A and B)

Calls `python -m src.generator.run_generator` for each OOD family. The generator respects the YAML's
`global.target_scenarios`; if you want fewer than that, set `MAX_OOD_SCENARIOS_PER_FAMILY` and the
downstream loops will subsample. Each generated scenario is a JSON in
`OOD_FAMILIES[fam]['scenarios_dir']` with the same schema as the in-distribution scenarios.

In [ ]:
if RUN_GENERATION:
    for fam, paths in OOD_FAMILIES.items():
        if paths['scenarios_dir'].exists() and any(paths['scenarios_dir'].glob('scenario_*.json')):
            print(f'[skip] {fam}: scenarios already present in {paths["scenarios_dir"]}')
            continue
        paths['scenarios_dir'].mkdir(parents=True, exist_ok=True)
        cmd = [sys.executable, '-m', 'src.generator.run_generator',
               '-c', str(paths['config_yaml']),
               '-o', str(paths['scenarios_dir']),
               '-v', 'v2']
        print('[run]', ' '.join(cmd))
        subprocess.run(cmd, cwd=str(REPO_PATH), check=True)
else:
    print('RUN_GENERATION=False; assuming OOD scenarios already exist on disk.')

for fam, paths in OOD_FAMILIES.items():
    n = len(list(paths['scenarios_dir'].glob('scenario_*.json'))) if paths['scenarios_dir'].exists() else 0
    print(f'  {fam}: {n} scenario JSON files in {paths["scenarios_dir"]}')

## 4. Reference (Generous-Budget) MILP for OOD Scenarios

Solves each OOD scenario with `time_limit_seconds = REFERENCE_TIME_LIMIT_SECONDS` to establish the
best-incumbent benchmark used as the reference for `*_gap_vs_reference_pct` later. Reports are saved
to `paths['reports_dir']/scenario_*.json` (same layout as `outputs/<crit>_criticality_scenarios/reports`).

**Heavy step.** ~`MAX_OOD_SCENARIOS_PER_FAMILY * REFERENCE_TIME_LIMIT_SECONDS` seconds in the worst
case. With defaults: 12 x 1200 s = 4 hours per family.

In [ ]:
def _serialize_summary(s):
    return {
        'objective': float(getattr(s, 'objective', float('nan'))),
        'termination': _termination_str(getattr(s, 'termination_condition', '')),
        'status': _termination_str(getattr(s, 'solver_status', '')),
        'solve_seconds': float(getattr(s, 'solve_seconds', float('nan')) or float('nan')),
    }

def _save_reference_report(payload, report_path):
    out = {
        'scenario_id': payload.get('scenario_id', report_path.stem),
        'mip': _serialize_summary(payload.get('mip')),
        'lp': _serialize_summary(payload.get('lp')),
        'cost_components': payload.get('cost_components', {}),
        'time_limit_seconds': float(REFERENCE_TIME_LIMIT_SECONDS),
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open('w', encoding='utf-8') as f:
        json.dump(out, f, indent=2)

if RUN_REFERENCE_MILP:
    for fam, paths in OOD_FAMILIES.items():
        scenario_paths = sorted(paths['scenarios_dir'].glob('scenario_*.json'))[:MAX_OOD_SCENARIOS_PER_FAMILY]
        for scenario_path in tqdm(scenario_paths, desc=f'reference MILP [{fam}]'):
            report_path = paths['reports_dir'] / scenario_path.name
            if report_path.exists():
                continue
            payload, elapsed = run_constrained_milp_with_guard(scenario_path, REFERENCE_TIME_LIMIT_SECONDS)
            _save_reference_report(payload, report_path)

for fam, paths in OOD_FAMILIES.items():
    n_rep = len(list(paths['reports_dir'].glob('scenario_*.json'))) if paths['reports_dir'].exists() else 0
    print(f'  {fam}: {n_rep} reports in {paths["reports_dir"]}')

## 5. Pipeline Run on OOD Scenarios (Full Repair Cascade)

This is the deployable pipeline (full feasibility cascade enabled). Question: does the EBM
trained on the in-distribution criticality space transfer to OOD topologies / adversarial constraints?

In [ ]:
ood_pipeline_config = PipelineConfig(
    repo_path=str(REPO_PATH),
    encoder_path=ENCODER_CHECKPOINT_REL,
    ebm_path=EBM_CHECKPOINT_REL,

    node_feature_dim=14, hidden_dim=128,
    num_spatial_layers=2, num_temporal_layers=4, num_heads=8,
    dropout=0.1, embed_dim=128, n_features=7, n_timesteps=24,

    langevin_steps=100, step_size=0.05, noise_scale=0.5,
    init_temp=1.0, final_temp=0.1,
    n_samples=5, init_mode='bernoulli',

    solver_name=SOLVER_NAME, slack_tol_mwh=1.0,
    flip_budget_repair_20=100, flip_budget_repair_100=1000,
    lp_worker_max_stages=5,                  # full cascade enabled
    lp_time_limit_hard_fix=300.0,
    lp_time_limit_repair_20=300.0,
    lp_time_limit_repair_100=300.0,
    lp_time_limit_full_soft=900.0,
    lp_time_limit_round_refix=900.0,
    lp_repair_poll_interval_seconds=5.0,

    strict_feasibility_tol_mwh=1.0,
    stop_on_first_strict_feasible=False,
    enable_exact_fallback=False,             # no MILP fallback (would defeat the point)

    device=DEVICE,
    seed=42,
)

if RUN_PIPELINE_OOD:
    runner_ood = PipelineRunner(ood_pipeline_config)
    runner_ood.load_models()
    for fam, paths in OOD_FAMILIES.items():
        results = runner_ood.evaluate_family(
            scenarios_dir=paths['scenarios_dir'],
            reports_dir=paths['reports_dir'],
            graphs_dir=OOD_OUTPUT_DIR / 'graphs',
            family_name=fam,
            max_scenarios=MAX_OOD_SCENARIOS_PER_FAMILY,
        )
        runner_ood.save_results(results, paths['pipeline_pkl'])
        print(f'[saved] {fam}: {len(results)} scenarios -> {paths["pipeline_pkl"]}')

ood_pipeline_results: Dict[str, List[Dict[str, Any]]] = {}
for fam, paths in OOD_FAMILIES.items():
    if paths['pipeline_pkl'].exists():
        with paths['pipeline_pkl'].open('rb') as f:
            ood_pipeline_results[fam] = pickle.load(f)
        backfill_projection_diagnostics(ood_pipeline_results[fam])
        print(f'  {fam}: {len(ood_pipeline_results[fam])} pipeline results loaded')
    else:
        ood_pipeline_results[fam] = []
        print(f'  {fam}: no pipeline cache')

## 6. Time-Constrained MILP on OOD Scenarios (Matched + Fixed)

Same protocol as `time_constrained_eval`: matched-pipeline (per-scenario budget = pipeline
wall-clock) plus a coarser fixed grid `FIXED_BUDGETS_SECONDS`.

In [ ]:
def _build_pipeline_keyed(pipeline_results, fam):
    out = {}
    for r in pipeline_results:
        sid = r.get('scenario_id') or r.get('id')
        out[(fam, sid)] = r
    return out

if RUN_MATCHED_MILP:
    for fam, paths in OOD_FAMILIES.items():
        if not ood_pipeline_results.get(fam):
            print(f'[skip] {fam}: no pipeline results')
            continue
        pipe_keyed = _build_pipeline_keyed(ood_pipeline_results[fam], fam)
        scenario_paths = sorted(paths['scenarios_dir'].glob('scenario_*.json'))[:MAX_OOD_SCENARIOS_PER_FAMILY]
        ref_reports = load_milp_reports(paths['reports_dir'])
        runs = []
        for sp in tqdm(scenario_paths, desc=f'tc MILP [{fam}]'):
            sid = sp.stem
            pr = pipe_keyed.get((fam, sid))
            if pr is None:
                continue
            ref = ref_reports.get(sid, {})
            ref_obj = _safe_float(ref.get('mip', {}).get('objective', float('nan')))
            ref_t = _safe_float(ref.get('mip', {}).get('solve_seconds', float('nan')))
            crit = _safe_float(pr.get('criticality_index', float('nan')))
            pipeline_time = _safe_float(pr.get('time_total', float('nan')), MIN_MATCHED_BUDGET_SECONDS)
            matched_budget = max(MIN_MATCHED_BUDGET_SECONDS, pipeline_time)
            pipeline_row = {
                'pipeline_objective': pr.get('lp_objective', float('nan')),
                'pipeline_solve_time': pipeline_time,
                'pipeline_best_status': pr.get('best_status', ''),
                'pipeline_stage_reached': pr.get('lp_stage_used', ''),
            }
            budgets = [('fixed', f'fixed_{b:g}s', float(b)) for b in FIXED_BUDGETS_SECONDS]
            budgets.append(('matched_pipeline', 'matched_pipeline_solve_time', float(matched_budget)))
            for mode, label, bs in budgets:
                payload, elapsed = run_constrained_milp_with_guard(sp, bs)
                runs.append(summarize_milp_run(
                    payload, family=fam, scenario_id=sid, criticality_index=crit,
                    reference_objective=ref_obj, reference_solve_time=ref_t,
                    budget_mode=mode, budget_label=label, budget_seconds=bs,
                    end_to_end_seconds=elapsed, pipeline_row=pipeline_row,
                ))
        with paths['tc_milp_pkl'].open('wb') as f:
            pickle.dump(runs, f)
        print(f'[saved] {fam}: {len(runs)} tc-MILP runs -> {paths["tc_milp_pkl"]}')

ood_tc_runs: Dict[str, List[Dict[str, Any]]] = {}
for fam, paths in OOD_FAMILIES.items():
    if paths['tc_milp_pkl'].exists():
        with paths['tc_milp_pkl'].open('rb') as f:
            ood_tc_runs[fam] = pickle.load(f)
        print(f'  {fam}: {len(ood_tc_runs[fam])} tc rows')
    else:
        ood_tc_runs[fam] = []

## 7. Experiment A and B Summary

For each OOD family, build the matched-budget table and the MILP time-to-first-feasible diagnostic.
If the pipeline can win anywhere, it will show up here as `pipeline_only_rate_pct > 0` or
`pipe_first_pct > 0`.

In [ ]:
def _ood_diagnostics(runs_list, family_label):
    if not runs_list:
        print(f'{family_label}: no runs')
        return None, None
    df = pd.DataFrame(runs_list)

    matched = build_budget_summary(df, 'matched_pipeline')
    fixed = build_budget_summary(df, 'fixed')

    # MILP TTF over fixed grid + right-censoring fallback to matched.
    fdf = df[df['budget_mode'] == 'fixed']
    mdf = df[df['budget_mode'] == 'matched_pipeline']
    rows = []
    for (fam, sid), grp in fdf.groupby(['family', 'scenario_id']):
        feas = grp[grp['milp_found_feasible']]
        ttf = float('nan') if not len(feas) else float(feas['budget_seconds'].min())
        rows.append({'family': fam, 'scenario_id': sid, 'milp_first_feasible_budget_s': ttf})
    ttf_df = pd.DataFrame(rows).merge(
        mdf[['family', 'scenario_id', 'pipeline_solve_time', 'pipeline_feasible_within_budget',
             'milp_found_feasible']].rename(columns={'milp_found_feasible': 'milp_feasible_at_matched'}),
        on=['family', 'scenario_id'], how='left',
    )
    max_fixed = float(max(FIXED_BUDGETS_SECONDS))
    needs_fb = ttf_df['milp_first_feasible_budget_s'].isna() & ttf_df['milp_feasible_at_matched'].fillna(False)
    ttf_df.loc[needs_fb, 'milp_first_feasible_budget_s'] = ttf_df.loc[needs_fb, 'pipeline_solve_time'].clip(lower=max_fixed)
    ttf_df['ratio'] = ttf_df['milp_first_feasible_budget_s'] / ttf_df['pipeline_solve_time'].replace(0, np.nan)
    ttf_df['pipe_first'] = ttf_df['milp_first_feasible_budget_s'].isna() | (ttf_df['pipeline_solve_time'] < ttf_df['milp_first_feasible_budget_s'])

    print(f'\n========== {family_label} ==========')
    print('\nMatched-budget summary:')
    print(matched.round(3).to_string(index=False))
    print('\nFixed-budget summary (ALL only):')
    print(fixed[fixed['family'] == 'ALL'].round(3).to_string(index=False))
    print('\nMILP time-to-first-feasible vs pipeline wall-clock:')
    print(ttf_df.groupby('family').agg(
        n=('scenario_id', 'count'),
        milp_ttf_median=('milp_first_feasible_budget_s', 'median'),
        pipe_time_median=('pipeline_solve_time', 'median'),
        ratio_median=('ratio', 'median'),
        pipe_first_pct=('pipe_first', lambda s: 100 * s.mean()),
    ).round(3))
    return matched, ttf_df

ood_summaries = {}
for fam in OOD_FAMILIES:
    ood_summaries[fam] = _ood_diagnostics(ood_tc_runs.get(fam, []), f'Experiment {"A" if fam == "hard" else "B"}: {fam}')

# Persist summaries
for fam, (matched, ttf) in ood_summaries.items():
    if matched is not None:
        matched.to_csv(OOD_OUTPUT_DIR / f'matched_summary_{fam}.csv', index=False)
        ttf.to_csv(OOD_OUTPUT_DIR / f'ttf_{fam}.csv', index=False)

## 8. Experiment C -- Lean Pipeline at Extreme Budgets

Build a stripped-down pipeline (`n_samples=1`, 20 Langevin steps, only `hard_fix` projection, no MILP
fallback). Compare to MILP at very small fixed budgets. The hypothesis is the lean pipeline can finish
in ~1 s while MILP cannot even finish presolve at 1 s on the medium/high in-distribution scenarios.

In [ ]:
lean_pipeline_config = PipelineConfig(
    repo_path=str(REPO_PATH),
    encoder_path=ENCODER_CHECKPOINT_REL,
    ebm_path=EBM_CHECKPOINT_REL,

    node_feature_dim=14, hidden_dim=128,
    num_spatial_layers=2, num_temporal_layers=4, num_heads=8,
    dropout=0.1, embed_dim=128, n_features=7, n_timesteps=24,

    langevin_steps=20,                      # 5x fewer EBM steps
    step_size=0.08, noise_scale=0.5,
    init_temp=1.0, final_temp=0.1,
    n_samples=1,                            # K = 1
    init_mode='bernoulli',

    solver_name=SOLVER_NAME, slack_tol_mwh=1.0,
    flip_budget_repair_20=0, flip_budget_repair_100=0,
    lp_worker_max_stages=1,                 # hard_fix only
    lp_time_limit_hard_fix=10.0,            # tight LP cap
    lp_time_limit_repair_20=0.0,
    lp_time_limit_repair_100=0.0,
    lp_time_limit_full_soft=0.0,
    lp_time_limit_round_refix=0.0,
    lp_repair_poll_interval_seconds=2.0,

    strict_feasibility_tol_mwh=1.0,
    stop_on_first_strict_feasible=True,
    enable_exact_fallback=False,

    device=DEVICE,
    seed=42,
)

LEAN_OUTPUT_DIR = OOD_OUTPUT_DIR / 'experiment_c_lean'
LEAN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LEAN_PIPELINE_PKL = LEAN_OUTPUT_DIR / 'lean_pipeline_results.pkl'
LEAN_TC_MILP_PKL = LEAN_OUTPUT_DIR / 'lean_tc_milp_runs.pkl'

if RUN_LEAN_PIPELINE:
    runner_lean = PipelineRunner(lean_pipeline_config)
    runner_lean.load_models()
    all_lean: List[PipelineResult] = []
    for fam, scenarios_dir in IND_FAMILIES.items():
        reports_dir = scenarios_dir / 'reports'
        results = runner_lean.evaluate_family(
            scenarios_dir=scenarios_dir,
            reports_dir=reports_dir,
            graphs_dir=LEAN_OUTPUT_DIR / 'graphs',
            family_name=fam,
            max_scenarios=MAX_OOD_SCENARIOS_PER_FAMILY,
        )
        all_lean.extend(results)
    runner_lean.save_results(all_lean, LEAN_PIPELINE_PKL)
    print(f'[saved] lean pipeline: {len(all_lean)} -> {LEAN_PIPELINE_PKL}')

if LEAN_PIPELINE_PKL.exists():
    with LEAN_PIPELINE_PKL.open('rb') as f:
        lean_results = pickle.load(f)
    backfill_projection_diagnostics(lean_results)
    print(f'Lean pipeline cache: {len(lean_results)} scenarios')
    lean_times = pd.Series([r.get('time_total', float('nan')) for r in lean_results])
    print(f'Lean wall-clock: median={lean_times.median():.2f}s P10={lean_times.quantile(0.10):.2f}s P90={lean_times.quantile(0.90):.2f}s')
else:
    lean_results = []
    print('No lean pipeline cache.')

In [ ]:
# Tiny-budget MILP for Experiment C
if RUN_LEAN_MILP and lean_results:
    lean_keyed = {(r.get('family'), r.get('scenario_id') or r.get('id')): r for r in lean_results}
    runs = []
    for fam, scenarios_dir in IND_FAMILIES.items():
        reports_dir = scenarios_dir / 'reports'
        ref_reports = load_milp_reports(reports_dir)
        scenario_paths = sorted(scenarios_dir.glob('scenario_*.json'))[:MAX_OOD_SCENARIOS_PER_FAMILY]
        for sp in tqdm(scenario_paths, desc=f'lean tc MILP [{fam}]'):
            sid = sp.stem
            pr = lean_keyed.get((fam, sid))
            if pr is None:
                continue
            ref = ref_reports.get(sid, {})
            ref_obj = _safe_float(ref.get('mip', {}).get('objective', float('nan')))
            ref_t = _safe_float(ref.get('mip', {}).get('solve_seconds', float('nan')))
            crit = _safe_float(pr.get('criticality_index', float('nan')))
            pt = _safe_float(pr.get('time_total', float('nan')), MIN_MATCHED_BUDGET_SECONDS)
            matched_budget = max(MIN_MATCHED_BUDGET_SECONDS, pt)
            pipeline_row = {
                'pipeline_objective': pr.get('lp_objective', float('nan')),
                'pipeline_solve_time': pt,
                'pipeline_best_status': pr.get('best_status', ''),
                'pipeline_stage_reached': pr.get('lp_stage_used', ''),
            }
            budgets = [('fixed', f'fixed_{b:g}s', float(b)) for b in LEAN_FIXED_BUDGETS_SECONDS]
            budgets.append(('matched_pipeline', 'matched_lean_pipeline', float(matched_budget)))
            for mode, label, bs in budgets:
                payload, elapsed = run_constrained_milp_with_guard(sp, bs)
                runs.append(summarize_milp_run(
                    payload, family=fam, scenario_id=sid, criticality_index=crit,
                    reference_objective=ref_obj, reference_solve_time=ref_t,
                    budget_mode=mode, budget_label=label, budget_seconds=bs,
                    end_to_end_seconds=elapsed, pipeline_row=pipeline_row,
                ))
    with LEAN_TC_MILP_PKL.open('wb') as f:
        pickle.dump(runs, f)
    print(f'[saved] lean tc MILP -> {LEAN_TC_MILP_PKL}')

if LEAN_TC_MILP_PKL.exists():
    with LEAN_TC_MILP_PKL.open('rb') as f:
        lean_runs = pickle.load(f)
    print('Experiment C summaries:')
    lean_matched, lean_ttf = _ood_diagnostics(lean_runs, 'Experiment C (lean pipeline, in-distribution)')
    if lean_matched is not None:
        lean_matched.to_csv(LEAN_OUTPUT_DIR / 'matched_summary_lean.csv', index=False)
        lean_ttf.to_csv(LEAN_OUTPUT_DIR / 'ttf_lean.csv', index=False)
else:
    lean_matched, lean_ttf = None, None
    print('No lean tc-MILP cache.')

## 9. Aggregate Conclusions

Decision rule:

- **H-A is supported** if `pipeline_only_rate_pct > 0` and `milp_only_rate_pct < pipeline_only_rate_pct`
  on the `hard` family.
- **H-B is supported** under the same rule on the `adversarial` family.
- **H-C is supported** if at any of the lean fixed budgets the lean pipeline's feasible rate exceeds
  MILP's, with non-trivial absolute counts (>= 5 scenarios).

If none of the three triggers fire, the pipeline does not have a tractability niche over MILP at any
wall-clock the pipeline can fit in -- the negative result generalizes.

In [ ]:
verdict = {
    'experiment_A_hard': None,
    'experiment_B_adversarial': None,
    'experiment_C_lean': None,
}

def _verdict_from_matched(matched_df, label):
    if matched_df is None or len(matched_df) == 0:
        return f'{label}: no data'
    all_rows = matched_df[matched_df['family'] == 'ALL']
    if not len(all_rows):
        all_rows = matched_df
    r = all_rows.iloc[0]
    pipe_only = float(r.get('pipeline_only_rate_pct', 0.0))
    milp_only = float(r.get('milp_only_rate_pct', 0.0))
    pipe_better = float(r.get('pipeline_better_gap_rate_pct', 0.0))
    n = int(r.get('n_scenarios', 0))
    supported = (pipe_only > 0) and (pipe_only > milp_only)
    return {
        'label': label,
        'n_scenarios': n,
        'pipeline_only_pct': pipe_only,
        'milp_only_pct': milp_only,
        'pipeline_better_gap_pct': pipe_better,
        'hypothesis_supported': bool(supported),
    }

verdict['experiment_A_hard'] = _verdict_from_matched(
    ood_summaries.get('hard', (None,))[0], 'A: structural-scale OOD (hard)'
)
verdict['experiment_B_adversarial'] = _verdict_from_matched(
    ood_summaries.get('adversarial', (None,))[0], 'B: adversarial UC (adversarial)'
)
verdict['experiment_C_lean'] = _verdict_from_matched(
    lean_matched, 'C: lean pipeline at tiny budgets'
)

print(json.dumps(verdict, indent=2, default=str))
with (OOD_OUTPUT_DIR / 'verdict.json').open('w', encoding='utf-8') as f:
    json.dump(verdict, f, indent=2, default=str)
print(f"Saved verdict to {OOD_OUTPUT_DIR / 'verdict.json'}")